# Cross-checking `future` against the legacy Fortran solver

The L-shape multipatch assembly built in `10_lshape_multipatch.ipynb` (2 squares with
holes, 2 corner sweep patches, 1 vertical-arm pair of squares) is reused here, **without**
any of that notebook's h-refinement exercises, to cross-check `future`'s
`PatchIntegrator.assemble_stiffness()` against the legacy Fortran assembly
(`sys_linmat_lindef_static`), reading its model from `.inp`/`.NB` text files.

To make the two matrices comparable **entry for entry** (no permutation), the legacy
model is generated directly from the `future` model's own global CP numbering:
coordinates and weights become the legacy `*Node` list verbatim, and each patch's
`.spans()` + `.global_indices` are turned into `*Nijk`/`*Weight`/`*Element` entries in
legacy's 1-based, u-fastest-reversed convention.

In [ ]:
import os
import time
import tempfile

import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp

from yeti_iga.future.bspline import (
    BSpline, BSplineSurface, ControlPointManager,
    Patch, GlobalDOFManager, PatchDOFManager, PatchAssembly,
    PatchIntegrator, Material, PlaneStress, PRefiner, SubdivisionRefiner,
)
from yeti_iga.future.refinement import refine_from, refine_assembly_uniform

from yeti_iga.preprocessing.igaparametrization import IGAparametrization
from yeti_iga.stiffmtrx_elemstorage import sys_linmat_lindef_static as build_stiffmatrix_legacy

## A legacy limitation found along the way

Bisecting an initial `free(): invalid pointer` crash (single patches worked; two patches
sharing CPs, and even two *unrelated* patches, both crashed) isolated the exact trigger:
**any patch with an asymmetric degree (u-degree ≠ v-degree) inside a *multi*-patch
assembly** corrupts memory in the legacy routine — confirmed by toggling degree
combinations in isolation (`[2,2]+[2,2]` OK, `[1,1]+[1,1]` OK, `[2,1]+[2,1]` and
`[2,1]+[2,2]` both crash, regardless of CP sharing). Every quarter-patch of the L-shape's
squares is degree `[2,1]` (angular × radial), so the model can't be pushed through this
legacy code path as-is. This looks like a genuine, previously untested edge case in the
legacy multi-patch assembly rather than a data issue on the `future` side — worth a look
separately.

**Workaround**: all 16 square quarter-patches are degree-elevated from `[2,1]` to `[2,2]`
(matching the corner patches) via `PRefiner` + `refine_from` right after building the
geometry, *before* any comparison — sidestepping the asymmetric-degree trigger while
keeping the same geometry and holes. No further h-refinement is applied in this notebook.

## Geometry — same L-shape as `10_lshape_multipatch.ipynb`

4 diagonal-cut squares (A, B: horizontal arm; C, D: vertical arm) + 2 corner sweep
patches, 67 control points, 18 patches. See that notebook for the full derivation of each
control point; the construction is reproduced here unchanged.

In [ ]:
R  = 0.25;  L  = 1.0
w_c = 1.0 / np.sqrt(2.0)
r_d = R / np.sqrt(2.0)
r_a = R * np.sqrt(2.0)

mgr = ControlPointManager(dim=2)

# ── Square A: center (0, 0) ───────────────────────────────────────────────────────────────────
mgr.add_point([ r_d, -r_d], w=1.0)
mgr.add_point([ r_d,  r_d], w=1.0)
mgr.add_point([-r_d,  r_d], w=1.0)
mgr.add_point([-r_d, -r_d], w=1.0)
mgr.add_point([ L,   -L  ], w=1.0)
mgr.add_point([ L,    L  ], w=1.0)
mgr.add_point([-L,    L  ], w=1.0)
mgr.add_point([-L,   -L  ], w=1.0)
mgr.add_point([ r_a,  0.0], w=w_c)
mgr.add_point([ L,    0.0], w=1.0)
mgr.add_point([ 0.0,  r_a], w=w_c)
mgr.add_point([ 0.0,  L  ], w=1.0)
mgr.add_point([-r_a,  0.0], w=w_c)
mgr.add_point([-L,    0.0], w=1.0)
mgr.add_point([ 0.0, -r_a], w=w_c)
mgr.add_point([ 0.0, -L  ], w=1.0)

# ── Square B: center (2L, 0) ────────────────────────────────────────────────────────
cx = 2*L
mgr.add_point([cx+r_d, -r_d], w=1.0)
mgr.add_point([cx+r_d,  r_d], w=1.0)
mgr.add_point([cx-r_d,  r_d], w=1.0)
mgr.add_point([cx-r_d, -r_d], w=1.0)
mgr.add_point([ 3*L,   -L  ], w=1.0)
mgr.add_point([ 3*L,    L  ], w=1.0)
mgr.add_point([cx+r_a,  0.0], w=w_c)
mgr.add_point([ 3*L,    0.0], w=1.0)
mgr.add_point([ cx,     r_a], w=w_c)
mgr.add_point([ cx,     L  ], w=1.0)
mgr.add_point([cx-r_a,  0.0], w=w_c)
mgr.add_point([ cx,    -r_a], w=w_c)
mgr.add_point([ cx,    -L  ], w=1.0)

# ── Corner patch 1: sweep from A's left edge (45° CCW arc, centre (-1,-2)) ──────────
w_arc45  = np.cos(np.pi / 8)
D        = np.array([-1.0 - np.sin(np.pi/4), -2.0 + np.cos(np.pi/4)])
arc_apex1 = np.array([-np.sqrt(2.0), -1.0])
P2       = np.array([-4.0,  1.0])
top_mid  = np.array([-2.5,  1.0])
left_mid = 0.5 * (D + P2)
interior1 = (0.5 * (np.array([-1., 0.]) + left_mid)
           + 0.5 * (arc_apex1 + top_mid)
           - 0.25 * (np.array([-1.,-1.]) + np.array([-1., 1.]) + D + P2))
mgr.add_point(arc_apex1.tolist(), w=w_arc45)
mgr.add_point(D.tolist(),         w=1.0)
mgr.add_point(left_mid.tolist(),  w=1.0)
mgr.add_point(interior1.tolist(), w=1.0)
mgr.add_point(top_mid.tolist(),   w=1.0)
mgr.add_point(P2.tolist(),        w=1.0)

# ── Corner patch 2: symmetric to patch 1 about the line P2–D ────────────────────
arc_apex2  = np.array([-2.0, -3.0 + np.sqrt(2.0)])
bot_R      = np.array([-2.0, -2.0])
mid_right  = 0.5 * (P2 + np.array([-4., -2.]))
bot_L      = np.array([-4.0, -2.0])
mid_bot    = np.array([-3.0, -2.0])
interior2 = (0.5 * (arc_apex2 + mid_right)
           + 0.5 * (left_mid + mid_bot)
           - 0.25 * (D + bot_R + P2 + bot_L))
mgr.add_point(arc_apex2.tolist(), w=w_arc45)
mgr.add_point(bot_R.tolist(),     w=1.0)
mgr.add_point(mid_right.tolist(), w=1.0)
mgr.add_point(interior2.tolist(), w=1.0)
mgr.add_point(mid_bot.tolist(),   w=1.0)
mgr.add_point(bot_L.tolist(),     w=1.0)

# ── Square C: center (-3, -3) ── vertical arm ────────────────────────────────
cx_c, cy_c = -3.0, -3.0
mgr.add_point([cx_c+r_d, cy_c-r_d], w=1.0)
mgr.add_point([cx_c+r_d, cy_c+r_d], w=1.0)
mgr.add_point([cx_c-r_d, cy_c+r_d], w=1.0)
mgr.add_point([cx_c-r_d, cy_c-r_d], w=1.0)
mgr.add_point([cx_c+L,   cy_c-L  ], w=1.0)
mgr.add_point([cx_c-L,   cy_c-L  ], w=1.0)
mgr.add_point([cx_c+r_a, cy_c    ], w=w_c)
mgr.add_point([cx_c+L,   cy_c    ], w=1.0)
mgr.add_point([cx_c,     cy_c+r_a], w=w_c)
mgr.add_point([cx_c-r_a, cy_c    ], w=w_c)
mgr.add_point([cx_c-L,   cy_c    ], w=1.0)
mgr.add_point([cx_c,     cy_c-r_a], w=w_c)
mgr.add_point([cx_c,     cy_c-L  ], w=1.0)

# ── Square D: center (-3, -5) ── vertical arm ────────────────────────────────
cx_d, cy_d = -3.0, -5.0
mgr.add_point([cx_d+r_d, cy_d-r_d], w=1.0)
mgr.add_point([cx_d+r_d, cy_d+r_d], w=1.0)
mgr.add_point([cx_d-r_d, cy_d+r_d], w=1.0)
mgr.add_point([cx_d-r_d, cy_d-r_d], w=1.0)
mgr.add_point([cx_d+L,   cy_d-L  ], w=1.0)
mgr.add_point([cx_d-L,   cy_d-L  ], w=1.0)
mgr.add_point([cx_d+r_a, cy_d    ], w=w_c)
mgr.add_point([cx_d+L,   cy_d    ], w=1.0)
mgr.add_point([cx_d,     cy_d+r_a], w=w_c)
mgr.add_point([cx_d-r_a, cy_d    ], w=w_c)
mgr.add_point([cx_d-L,   cy_d    ], w=1.0)
mgr.add_point([cx_d,     cy_d-r_a], w=w_c)
mgr.add_point([cx_d,     cy_d-L  ], w=1.0)

def make_sq_surface():
    return BSplineSurface(BSpline(2, np.array([0., 0., 0., 1., 1., 1.])),
                          BSpline(1, np.array([0., 0., 1., 1.])))

def make_corner_surface():
    return BSplineSurface(BSpline(2, np.array([0., 0., 0., 1., 1., 1.])),
                          BSpline(2, np.array([0., 0., 0., 1., 1., 1.])))

all_maps = [
    # Square A (shape [3,2])
    [0,  8,  1,  4,  9,  5], [1, 10,  2,  5, 11,  6],
    [2, 12,  3,  6, 13,  7], [3, 14,  0,  7, 15,  4],
    # Square B (shape [3,2])
    [16, 22, 17, 20, 23, 21], [17, 24, 18, 21, 25,  5],
    [18, 26, 19,  5,  9,  4], [19, 27, 16,  4, 28, 20],
    # Corner 1, Corner 2 (shape [3,3])
    [7, 29, 30,  13, 32, 31,  6, 33, 34],
    [30, 31, 34,  35, 38, 37,  36, 39, 40],
    # Square C (shape [3,2])
    [41, 47, 42, 45, 48, 36], [42, 49, 43, 36, 39, 40],
    [43, 50, 44, 40, 51, 46], [44, 52, 41, 46, 53, 45],
    # Square D (shape [3,2])
    [54, 60, 55, 58, 61, 45], [55, 62, 56, 45, 53, 46],
    [56, 63, 57, 46, 64, 59], [57, 65, 54, 59, 66, 58],
]
shapes = [[3, 2]] * 8 + [[3, 3], [3, 3]] + [[3, 2]] * 8

dofs = 2
gm = GlobalDOFManager([dofs] * mgr.n_points)
patches = []
for m, sh in zip(all_maps, shapes):
    surf = make_corner_surface() if len(m) == 9 else make_sq_surface()
    pdm = PatchDOFManager(dofs, m, gm)
    patches.append(Patch(surf, mgr, m, sh, pdm))

assembly = PatchAssembly()
for p in patches:
    assembly.add_patch(p)
assembly.detect_shared_control_points()
assembly.detect_interfaces()

print(f'mgr.n_points = {mgr.n_points}  (expected 67)')
print(f'n_cp per patch: {[p.n_cp for p in patches]}')

## Degree elevation — squares [2,1] → [2,2]

One `PRefiner(direction=1, n_elevations=1)` trigger per square, propagated via
`refine_from` around that square's own 4-quadrant loop (same mechanism as h-refinement:
elevating v, the radial direction, propagates through the radial-spoke interfaces shared
between quadrants of the *same* square, but not across the flat outer edges shared with
another square or a corner patch).

In [ ]:
for start in [0, 4, 10, 14]:   # one quadrant per square (A, B, C, D)
    touched = refine_from(assembly, patches, start_index=start, start_direction=1,
                          refiner_factory=lambda d: PRefiner(direction=d, n_elevations=1))
    print(f'  elevated square starting at patch {start}: {touched}')

print('degrees after elevation:',
      set((p.tensor.components[0].degree, p.tensor.components[1].degree) for p in patches))

assembly.update_dof_managers(gm, dofs_per_cp=dofs)
assembly.compact()
assembly.detect_shared_control_points()
gm = GlobalDOFManager([dofs] * mgr.n_points)
assembly.update_dof_managers(gm, dofs_per_cp=dofs)
print(f'mgr.n_points = {mgr.n_points}, total dofs = {gm.n_control_points() * dofs}')

## `future` stiffness matrix — plane stress, steel

Classic steel: E = 210 GPa, ν = 0.3. `PatchIntegrator.assemble_stiffness()` builds one
`IGABasis1D` pair and integrator per patch internally and sums shared-dof contributions
automatically.

In [ ]:
steel = PlaneStress(Material(E=210e9, nu=0.3))
laws = [steel] * len(patches)

K_future = PatchIntegrator.assemble_stiffness(assembly, laws)
print(f'K_future.shape = {K_future.shape}, nnz = {K_future.nnz}')

## Bridge to legacy `.inp`/`.NB` files

Legacy conventions (reverse-engineered from the parser and a working multipatch fixture):
`*Nijk`/`*Weight` lines are read POSITIONALLY (the element id printed on the line is
ignored), grouped by patch in declaration order; `*Element` rows list a patch's active CPs
in u-fastest DECREASING order (the reverse of `future`'s u-fastest-increasing
`global_indices`); patches are grouped into `*ELSET,ELSET=EltPatchN` (1-based) for
material assignment, and distinct per-patch node counts each need their own
`*USER ELEMENT TYPE=` block.

In [ ]:
def write_legacy_files(mgr, patches, basename, E, nu):
    """Bridge a `future` multipatch model to legacy `.inp`/`.NB` files, reusing
    `mgr`'s global CP numbering verbatim so the resulting legacy dof numbering
    matches the `future` model exactly (no permutation needed to compare)."""
    coords_all = mgr.coords_view()
    weights_all = mgr.weights_view() if mgr.is_rational else np.ones(mgr.n_points)
    n_cp = mgr.n_points

    patch_data = []
    for patch in patches:
        su = patch.tensor.components[0]
        sv = patch.tensor.components[1]
        p_u, p_v = su.degree, sv.degree
        n_u = patch.local_shape[0]
        gidx = patch.global_indices
        nnode = (p_u + 1) * (p_v + 1)
        elements = []
        for span in patch.spans():
            span_u, span_v = span[0], span[1]
            window_u = range(span_u - p_u, span_u + 1)
            window_v = range(span_v - p_v, span_v + 1)
            Lw = [iu + n_u * iv for iv in window_v for iu in window_u]  # u-fastest increasing
            global_ids_inc = [gidx[k] for k in Lw]
            IEN_row = [gid + 1 for gid in reversed(global_ids_inc)]        # 1-based, reversed
            weight_row = [weights_all[gid] for gid in reversed(global_ids_inc)]
            elements.append(((span_u + 1, span_v + 1), IEN_row, weight_row))
        patch_data.append(dict(p_u=p_u, p_v=p_v, ku=su.knot_vector, kv=sv.knot_vector,
                                nnode=nnode, elements=elements))

    n_elem_by_patch = [len(pd['elements']) for pd in patch_data]

    nb = []
    nb.append('*Dimension'); nb.append(','.join(['2'] * len(patch_data)))
    nb.append('*Number of CP by element'); nb.append(','.join(str(pd['nnode']) for pd in patch_data))
    nb.append('*Number of patch'); nb.append(str(len(patch_data)))
    nb.append('*Total number of element'); nb.append(str(sum(n_elem_by_patch)))
    nb.append('*Number of element by patch'); nb.append(','.join(str(n) for n in n_elem_by_patch))
    for p_idx, pd in enumerate(patch_data):
        nb.append(f'*Patch({p_idx + 1})')
        nb.append(str(len(pd['ku']))); nb.append(','.join(f'{v:.16g}' for v in pd['ku']))
        nb.append(str(len(pd['kv']))); nb.append(','.join(f'{v:.16g}' for v in pd['kv']))
    nb.append('*Jpqr')
    for pd in patch_data:
        nb.append(f"{pd['p_u']},{pd['p_v']}")
    nb.append('*Nijk')
    eid = 0
    for pd in patch_data:
        for Nijk, IEN_row, w_row in pd['elements']:
            eid += 1
            nb.append(f'{eid},{Nijk[0]},{Nijk[1]}')
    nb.append('*Weight')
    eid = 0
    for pd in patch_data:
        for Nijk, IEN_row, w_row in pd['elements']:
            eid += 1
            nb.append(f"{eid}," + ','.join(f'{w:.16g}' for w in w_row))
    with open(f'{basename}.NB', 'w') as f:
        f.write('\n'.join(nb) + '\n')

    distinct_nnode = sorted(set(pd['nnode'] for pd in patch_data))
    type_of = {n: f'U{i + 1}' for i, n in enumerate(distinct_nnode)}

    inp = ['*HEADING', '*Part, name=Piece']
    for nnode in distinct_nnode:
        inp.append(f'*USER ELEMENT, NODES={nnode}, TYPE={type_of[nnode]}, COORDINATES=2, '
                    f'INTEGRATION={nnode}, TENSOR=PSTRESS')
        inp.append('1,2')
    inp.append('*Node,nset=AllNode')
    for i in range(n_cp):
        x, y = coords_all[i]
        inp.append(f'{i + 1}, {x:.16g}, {y:.16g}, 0.0')

    eid = 0
    elems_by_type = {t: [] for t in type_of.values()}
    elems_by_patch = []
    for pd in patch_data:
        t = type_of[pd['nnode']]
        ids = []
        for Nijk, IEN_row, w_row in pd['elements']:
            eid += 1
            ids.append(eid)
            elems_by_type[t].append((eid, IEN_row))
        elems_by_patch.append(ids)
    for t, rows in elems_by_type.items():
        inp.append(f'*Element,type={t},elset=AllEls{t}')
        for e, IEN_row in rows:
            inp.append(f'{e}, ' + ', '.join(str(n) for n in IEN_row))
    for p_idx, ids in enumerate(elems_by_patch):
        inp.append(f'*ELSET,ELSET=EltPatch{p_idx + 1}'); inp.append(','.join(str(i) for i in ids))
    for p_idx in range(len(patch_data)):
        inp.append(f'*UEL PROPERTY, ELSET=EltPatch{p_idx + 1}, MATERIAL=STEEL'); inp.append('1')
    inp += ['*End Part', '*Assembly, name=Assembly', '*Instance, name=I1, part=Piece',
            '*End Instance', '*End Assembly', '*MATERIAL,NAME=STEEL', '*Elastic',
            f'{E:.16g}, {nu}', '*STEP,extrapolation=NO,NLGEOM=NO', '*Static', '*End Step']
    with open(f'{basename}.inp', 'w') as f:
        f.write('\n'.join(inp) + '\n')


legacy_basename = os.path.join(tempfile.gettempdir(), 'lshape_legacy_crosscheck')
write_legacy_files(mgr, patches, legacy_basename, E=210e9, nu=0.3)
print('Legacy .inp/.NB files written to', legacy_basename)

In [ ]:
legacy_model = IGAparametrization(filename=legacy_basename)
print(f'legacy: nb_cp={legacy_model.nb_cp}, nb_dof_tot={legacy_model.nb_dof_tot}')
assert legacy_model.nb_dof_tot == K_future.shape[0], \
    'legacy and future dof counts must match for a direct comparison'

data, row, col, _ = build_stiffmatrix_legacy(*legacy_model.get_inputs4system_elemStorage())
K_side = sp.coo_matrix((data, (row, col)),
                        shape=(legacy_model.nb_dof_tot, legacy_model.nb_dof_tot),
                        dtype='float64').tocsc()
K_legacy = K_side + K_side.transpose()   # legacy stores only the upper triangle
print(f'K_legacy.shape = {K_legacy.shape}, nnz = {K_legacy.nnz}')

## Comparison

In [ ]:
Kf = K_future.toarray()
Kl = K_legacy.toarray()
diff = Kf - Kl

max_abs = np.abs(diff).max()
max_rel = max_abs / np.abs(Kf).max()
rel_fro_error = np.linalg.norm(diff) / np.linalg.norm(Kf)

print(f'max |K_future|            = {np.abs(Kf).max():.6e}')
print(f'max |K_future - K_legacy| = {max_abs:.6e}')
print(f'max relative difference   = {max_rel:.3e}')
print(f'relative Frobenius error  = {rel_fro_error:.3e}')
print(f'match (rel_fro_error < 1e-8)? {rel_fro_error < 1e-8}')

worst = np.unravel_index(np.argmax(np.abs(diff)), diff.shape)
print(f'worst entry at dof {worst}: future={Kf[worst]:.6f}, legacy={Kl[worst]:.6f}')

---

## Performance benchmark across refinement levels

Repeat the same future-vs-legacy comparison at increasing uniform h-refinement levels
(0, 1, 2 — same convention as `SubdivisionRefiner`'s `n_levels`, applied to *every* patch
in *both* parametric directions via `refine_assembly_uniform`), timing only the actual
stiffness-assembly call on each side:
- `future`: `PatchIntegrator.assemble_stiffness(assembly, laws)`
- `legacy`: `sys_linmat_lindef_static(*inputs)`

(model construction, degree elevation, file writing, and `.inp`/`.NB` parsing are excluded
from the timings — only the numerical assembly kernel itself is measured). Each level
rebuilds a fresh model (`build_lshape_model()` reproduces the geometry + degree-elevation
cells above) rather than incrementally refining, to avoid any stale-state risk between
levels, and re-verifies the two matrices still match before recording a data point.

In [ ]:
def build_lshape_model():
    """Rebuild the L-shape geometry (same as above) from scratch, already
    degree-elevated to [2,2] everywhere, ready for h-refinement at any level."""
    R = 0.25; L = 1.0
    w_c = 1.0 / np.sqrt(2.0)
    r_d = R / np.sqrt(2.0)
    r_a = R * np.sqrt(2.0)

    mgr = ControlPointManager(dim=2)

    mgr.add_point([ r_d, -r_d], w=1.0); mgr.add_point([ r_d,  r_d], w=1.0)
    mgr.add_point([-r_d,  r_d], w=1.0); mgr.add_point([-r_d, -r_d], w=1.0)
    mgr.add_point([ L,   -L  ], w=1.0); mgr.add_point([ L,    L  ], w=1.0)
    mgr.add_point([-L,    L  ], w=1.0); mgr.add_point([-L,   -L  ], w=1.0)
    mgr.add_point([ r_a,  0.0], w=w_c); mgr.add_point([ L,    0.0], w=1.0)
    mgr.add_point([ 0.0,  r_a], w=w_c); mgr.add_point([ 0.0,  L  ], w=1.0)
    mgr.add_point([-r_a,  0.0], w=w_c); mgr.add_point([-L,    0.0], w=1.0)
    mgr.add_point([ 0.0, -r_a], w=w_c); mgr.add_point([ 0.0, -L  ], w=1.0)

    cx = 2*L
    mgr.add_point([cx+r_d, -r_d], w=1.0); mgr.add_point([cx+r_d,  r_d], w=1.0)
    mgr.add_point([cx-r_d,  r_d], w=1.0); mgr.add_point([cx-r_d, -r_d], w=1.0)
    mgr.add_point([ 3*L,   -L  ], w=1.0); mgr.add_point([ 3*L,    L  ], w=1.0)
    mgr.add_point([cx+r_a,  0.0], w=w_c); mgr.add_point([ 3*L,    0.0], w=1.0)
    mgr.add_point([ cx,     r_a], w=w_c); mgr.add_point([ cx,     L  ], w=1.0)
    mgr.add_point([cx-r_a,  0.0], w=w_c); mgr.add_point([ cx,    -r_a], w=w_c)
    mgr.add_point([ cx,    -L  ], w=1.0)

    w_arc45 = np.cos(np.pi / 8)
    D = np.array([-1.0 - np.sin(np.pi/4), -2.0 + np.cos(np.pi/4)])
    arc_apex1 = np.array([-np.sqrt(2.0), -1.0])
    P2 = np.array([-4.0, 1.0])
    top_mid = np.array([-2.5, 1.0])
    left_mid = 0.5 * (D + P2)
    interior1 = (0.5 * (np.array([-1., 0.]) + left_mid) + 0.5 * (arc_apex1 + top_mid)
               - 0.25 * (np.array([-1.,-1.]) + np.array([-1., 1.]) + D + P2))
    mgr.add_point(arc_apex1.tolist(), w=w_arc45); mgr.add_point(D.tolist(), w=1.0)
    mgr.add_point(left_mid.tolist(), w=1.0); mgr.add_point(interior1.tolist(), w=1.0)
    mgr.add_point(top_mid.tolist(), w=1.0); mgr.add_point(P2.tolist(), w=1.0)

    arc_apex2 = np.array([-2.0, -3.0 + np.sqrt(2.0)])
    bot_R = np.array([-2.0, -2.0])
    mid_right = 0.5 * (P2 + np.array([-4., -2.]))
    bot_L = np.array([-4.0, -2.0])
    mid_bot = np.array([-3.0, -2.0])
    interior2 = (0.5 * (arc_apex2 + mid_right) + 0.5 * (left_mid + mid_bot)
               - 0.25 * (D + bot_R + P2 + bot_L))
    mgr.add_point(arc_apex2.tolist(), w=w_arc45); mgr.add_point(bot_R.tolist(), w=1.0)
    mgr.add_point(mid_right.tolist(), w=1.0); mgr.add_point(interior2.tolist(), w=1.0)
    mgr.add_point(mid_bot.tolist(), w=1.0); mgr.add_point(bot_L.tolist(), w=1.0)

    cx_c, cy_c = -3.0, -3.0
    mgr.add_point([cx_c+r_d, cy_c-r_d], w=1.0); mgr.add_point([cx_c+r_d, cy_c+r_d], w=1.0)
    mgr.add_point([cx_c-r_d, cy_c+r_d], w=1.0); mgr.add_point([cx_c-r_d, cy_c-r_d], w=1.0)
    mgr.add_point([cx_c+L,   cy_c-L  ], w=1.0); mgr.add_point([cx_c-L,   cy_c-L  ], w=1.0)
    mgr.add_point([cx_c+r_a, cy_c    ], w=w_c); mgr.add_point([cx_c+L,   cy_c    ], w=1.0)
    mgr.add_point([cx_c,     cy_c+r_a], w=w_c); mgr.add_point([cx_c-r_a, cy_c    ], w=w_c)
    mgr.add_point([cx_c-L,   cy_c    ], w=1.0); mgr.add_point([cx_c,     cy_c-r_a], w=w_c)
    mgr.add_point([cx_c,     cy_c-L  ], w=1.0)

    cx_d, cy_d = -3.0, -5.0
    mgr.add_point([cx_d+r_d, cy_d-r_d], w=1.0); mgr.add_point([cx_d+r_d, cy_d+r_d], w=1.0)
    mgr.add_point([cx_d-r_d, cy_d+r_d], w=1.0); mgr.add_point([cx_d-r_d, cy_d-r_d], w=1.0)
    mgr.add_point([cx_d+L,   cy_d-L  ], w=1.0); mgr.add_point([cx_d-L,   cy_d-L  ], w=1.0)
    mgr.add_point([cx_d+r_a, cy_d    ], w=w_c); mgr.add_point([cx_d+L,   cy_d    ], w=1.0)
    mgr.add_point([cx_d,     cy_d+r_a], w=w_c); mgr.add_point([cx_d-r_a, cy_d    ], w=w_c)
    mgr.add_point([cx_d-L,   cy_d    ], w=1.0); mgr.add_point([cx_d,     cy_d-r_a], w=w_c)
    mgr.add_point([cx_d,     cy_d-L  ], w=1.0)

    def make_sq_surface():
        return BSplineSurface(BSpline(2, np.array([0., 0., 0., 1., 1., 1.])),
                              BSpline(1, np.array([0., 0., 1., 1.])))

    def make_corner_surface():
        return BSplineSurface(BSpline(2, np.array([0., 0., 0., 1., 1., 1.])),
                              BSpline(2, np.array([0., 0., 0., 1., 1., 1.])))

    all_maps = [
        [0,  8,  1,  4,  9,  5], [1, 10,  2,  5, 11,  6],
        [2, 12,  3,  6, 13,  7], [3, 14,  0,  7, 15,  4],
        [16, 22, 17, 20, 23, 21], [17, 24, 18, 21, 25,  5],
        [18, 26, 19,  5,  9,  4], [19, 27, 16,  4, 28, 20],
        [7, 29, 30,  13, 32, 31,  6, 33, 34],
        [30, 31, 34,  35, 38, 37,  36, 39, 40],
        [41, 47, 42, 45, 48, 36], [42, 49, 43, 36, 39, 40],
        [43, 50, 44, 40, 51, 46], [44, 52, 41, 46, 53, 45],
        [54, 60, 55, 58, 61, 45], [55, 62, 56, 45, 53, 46],
        [56, 63, 57, 46, 64, 59], [57, 65, 54, 59, 66, 58],
    ]
    shapes = [[3, 2]] * 8 + [[3, 3], [3, 3]] + [[3, 2]] * 8

    dofs = 2
    gm = GlobalDOFManager([dofs] * mgr.n_points)
    patches = []
    for m, sh in zip(all_maps, shapes):
        surf = make_corner_surface() if len(m) == 9 else make_sq_surface()
        pdm = PatchDOFManager(dofs, m, gm)
        patches.append(Patch(surf, mgr, m, sh, pdm))

    assembly = PatchAssembly()
    for p in patches:
        assembly.add_patch(p)
    assembly.detect_shared_control_points()
    assembly.detect_interfaces()

    # degree elevation [2,1] -> [2,2] on the 16 square quadrants (legacy workaround)
    for start in [0, 4, 10, 14]:
        refine_from(assembly, patches, start_index=start, start_direction=1,
                   refiner_factory=lambda d: PRefiner(direction=d, n_elevations=1))

    assembly.update_dof_managers(gm, dofs_per_cp=dofs)
    assembly.compact()
    assembly.detect_shared_control_points()
    gm = GlobalDOFManager([dofs] * mgr.n_points)
    assembly.update_dof_managers(gm, dofs_per_cp=dofs)

    return mgr, patches, assembly, gm, dofs

In [ ]:
levels = [0, 1, 2, 3, 4, 5]
n_repeats = 3   # keep the min over a few repeats to reduce timing noise
bench_results = []   # (level, n_dofs, t_future, t_legacy, matrices_match)

for level in levels:
    mgr_i, patches_i, assembly_i, gm_i, dofs_i = build_lshape_model()

    if level > 0:
        refine_assembly_uniform(
            assembly_i, patches_i, directions=(0, 1),
            refiner_factory=lambda d: SubdivisionRefiner(direction=d, n_levels=level))
        assembly_i.update_dof_managers(gm_i, dofs_per_cp=dofs_i)
        assembly_i.compact()
        assembly_i.detect_shared_control_points()
        gm_i = GlobalDOFManager([dofs_i] * mgr_i.n_points)
        assembly_i.update_dof_managers(gm_i, dofs_per_cp=dofs_i)

    n_dofs = gm_i.n_control_points() * dofs_i
    laws_i = [steel] * len(patches_i)

    # --- future: time only the assembly call ---
    times_future = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        K_future_i = PatchIntegrator.assemble_stiffness(assembly_i, laws_i)
        times_future.append(time.perf_counter() - t0)
    t_future = min(times_future)

    # --- legacy: bridge, then time only the assembly call ---
    basename_i = os.path.join(tempfile.gettempdir(), f'lshape_bench_level{level}')
    write_legacy_files(mgr_i, patches_i, basename_i, E=210e9, nu=0.3)
    legacy_model_i = IGAparametrization(filename=basename_i)
    assert legacy_model_i.nb_dof_tot == n_dofs

    inputs_i = legacy_model_i.get_inputs4system_elemStorage()
    times_legacy = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        data, row, col, _ = build_stiffmatrix_legacy(*inputs_i)
        times_legacy.append(time.perf_counter() - t0)
    t_legacy = min(times_legacy)

    K_side_i = sp.coo_matrix((data, (row, col)), shape=(n_dofs, n_dofs), dtype='float64').tocsc()
    K_legacy_i = K_side_i + K_side_i.transpose()

    # Relative Frobenius-norm error (sparse, never densifies -- important once
    # n_dofs gets into the thousands) rather than an elementwise np.allclose:
    # a fixed atol works for small matrices but becomes too strict as
    # near-zero entries accumulate more floating-point round-off (still
    # utterly negligible relative to the matrix's own scale, ~1e11-1e12 here)
    # at finer meshes.
    rel_fro_error = sp.linalg.norm(K_future_i - K_legacy_i) / sp.linalg.norm(K_future_i)
    match = rel_fro_error < 1e-8
    bench_results.append((level, n_dofs, t_future, t_legacy, match))
    print(f'level={level}  n_dofs={n_dofs:5d}  '
          f't_future={t_future*1e3:8.3f} ms  t_legacy={t_legacy*1e3:8.3f} ms  '
          f'rel_fro_error={rel_fro_error:.2e}  match={match}')

assert all(r[4] for r in bench_results), 'future and legacy stiffness matrices must match at every level'

In [ ]:
levels_arr, ndofs_arr, tf_arr, tl_arr, _ = zip(*bench_results)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(ndofs_arr, np.array(tf_arr) * 1e3, 'o-', label='future (PatchIntegrator)')
ax.plot(ndofs_arr, np.array(tl_arr) * 1e3, 's-', label='legacy (Fortran)')
for lvl, n in zip(levels_arr, ndofs_arr):
    ax.annotate(f'level {lvl}', (n, tf_arr[levels_arr.index(lvl)] * 1e3),
                textcoords='offset points', xytext=(6, 6), fontsize=8)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Number of DOFs')
ax.set_ylabel('Stiffness assembly time (ms)')
ax.set_title('Stiffness assembly time vs. problem size')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()